## Simulador de Propriedades PVT para Engenharia Química
### Comparativo de Equações de Estado: RK vs SRK vs PR
**Desenvolvido por:** Matheus Wenner- Engenharia Química
---
*Este notebook realiza o cálculo do fator de compressibilidade (Z) para hidrocarbonetos leves e contaminantes, 
comparando modelos cúbicos fundamentais para a indústria de óleo e gás.*

In [71]:
# 1
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import json
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, clear_output

In [72]:
# 2 Garante que o gráfico apareça dentro do VS Code
pio.renderers.default = "vscode"
layout_padrao = dict(template="simple_white", font=dict(family="Arial", size=12))

In [73]:
# 3 Tenta carregar o arquivo externo que criamos
try:
    with open('fluidos.json', 'r') as f:
        fluidos = json.load(f)
    print(f"✅ Banco de dados carregado: {len(fluidos)} componentes prontos.")
except FileNotFoundError:
    print("❌ Erro: Certifique-se de que o arquivo 'fluidos.json' está na mesma pasta!")

✅ Banco de dados carregado: 13 componentes prontos.


In [74]:
# 4
caixa_diagnostico = widgets.Output()

In [75]:
# 5
def resolver_cubica_estavel(coefs, B):
    """Resolve a equação cúbica e filtra a raiz estável (fase vapor) Z > B."""
    raizes = np.roots(coefs)
    z_reais = raizes[np.isreal(raizes)].real
    # Filtro de segurança, em que Z deve ser maior que o co-volume B
    z_validos = z_reais[z_reais > B]
    return np.max(z_validos) if len(z_validos) > 0 else np.nan

def calcular_modelos_puros(fluido_nome, T_c):
    p = fluidos[fluido_nome]
    T = T_c + 273.15
    R = 8.314
    Tr = T / p["Tc"]
    w = p["omega"]
    
    pressões_mpa = np.linspace(0.1, 100, 300)  # Pressão de 0.1 MPa a 100 MPa, com 300 pontos
    # Escolhi o range de 100 MPa porque, na minha visão como futuro engenheiro, é essencial entender o comportamento do fluido nas condições extremas do Pré-Sal, 
    # e não apenas em condições ideais de laboratório. Qual é o contraponto verificado? Com um range tão grande, o simulador não irá mais mostrar domínio de atração,
    # sendo que ele foi programado para diagnosticar esse comportamento. Se quiser verificar, mude o 100 para 50 e o CO2 a 200 graus Celsius.
    z_rk, z_srk, z_pr = [], [], []

    for P_mpa in pressões_mpa:
        P = P_mpa * 1e6
        
        # --- REDLICH-KWONG (RK) ---
        a_rk = 0.42748 * (R**2 * p["Tc"]**2.5) / (p["Pc"] * np.sqrt(T))
        b_rk = 0.08664 * (R * p["Tc"]) / p["Pc"]
        A_rk, B_rk = (a_rk * P)/(R**2 * T**2), (b_rk * P)/(R * T)
        z_rk.append(resolver_cubica_estavel([1, -1, (A_rk - B_rk - B_rk**2), -A_rk * B_rk], B_rk))

        # --- SOAVE-REDLICH-KWONG (SRK) ---
        m_srk = 0.480 + 1.574*w - 0.176*w**2
        alpha_srk = (1 + m_srk * (1 - np.sqrt(Tr)))**2
        a_srk = (0.42748 * R**2 * p["Tc"]**2 / p["Pc"]) * alpha_srk
        b_srk = 0.08664 * R * p["Tc"] / p["Pc"]
        A_srk, B_srk = (a_srk * P)/(R**2 * T**2), (b_srk * P)/(R * T)
        z_srk.append(resolver_cubica_estavel([1, -1, (A_srk - B_srk - B_srk**2), -A_srk * B_srk], B_srk))

        # --- PENG-ROBINSON (PR) ---
        m_pr = 0.37464 + 1.54226*w - 0.26992*w**2
        alpha_pr = (1 + m_pr * (1 - np.sqrt(Tr)))**2
        a_pr = (0.45724 * R**2 * p["Tc"]**2 / p["Pc"]) * alpha_pr
        b_pr = 0.07780 * R * p["Tc"] / p["Pc"]
        A_pr, B_pr = (a_pr * P)/(R**2 * T**2), (b_pr * P)/(R * T)
        z_pr.append(resolver_cubica_estavel([1, -(1-B_pr), (A_pr - 2*B_pr - 3*B_pr**2), -(A_pr*B_pr - B_pr**2 - B_pr**3)], B_pr))
        
    return pressões_mpa, z_rk, z_srk, z_pr

In [76]:
# 6     

caixa_diagnostico = widgets.Output()

teoria_z = widgets.HTML("""
<div style="background-color: #f7f9fc; padding: 25px; border-radius: 12px; margin-top: 25px; font-family: 'Segoe UI', serif; border: 1px solid #dce4ec; border-left: 6px solid #1a2a40;">
    <h2 style="color: #1a2a40; margin-top: 0; font-size: 1.5em; border-bottom: 2px solid #1a2a40; padding-bottom: 5px;">📘 Fundamentos: O Fator de Compressibilidade (Z)</h2>
    
    <div style="display: flex; gap: 20px; margin-top: 15px;">
        <div style="flex: 1; border-right: 1px solid #dce4ec; padding-right: 15px;">
            <b style="color: #1a2a40; font-size: 1.1em;">O que é Z?</b><br>
            Quantifica o desvio de um gás real em relação ao comportamento ideal. Para um Gás Ideal, <b>Z = 1</b>.<br>
            <i style="color: #555;">(Z = PV / RT)</i>
        </div>
        
        <div style="flex: 1; border-right: 1px solid #dce4ec; padding-right: 15px;">
            <b style="color: #d4a017; font-size: 1.1em;">Z < 1 (Domínio de Atração)</b><br>
            As forças atrativas intermoleculares são predominantes, reduzindo o volume real.
        </div>
        
        <div style="flex: 1;">
            <b style="color: #b22222; font-size: 1.1em;">Z > 1 (Domínio de Repulsão)</b><br>
            O co-volume físico das moléculas impede maior compressão. As forças repulsivas dominam (ex: HPHT).
        </div>
    </div>
    
    <hr style="border: 0; border-top: 1px solid #dce4ec; margin: 20px 0;">
    
    <p style="font-size: 0.85em; color: #666; margin-bottom: 0;">
        <b>📌 Rastreabilidade e Limitações (Technical Note):</b><br>
        <i>Propriedades críticas (Tc, Pc, ω) extraídas de:</i><br>
        1. <i>Smith, Van Ness & Abbott - Introduction to Chemical Engineering Thermodynamics (7th Ed.)</i><br>
        2. Banco de Dados Oficial do <i>NIST Chemistry WebBook</i> (National Institute of Standards and Technology).<br>
        <br>
        <b style="color: #1a2a40;">⚠️ Nota sobre Colunas de Destilação:</b> Os modelos Peng-Robinson (PR), RK e SRK são excelentes para fase vapor e misturas não-polares. <br>
        Para fundo de colunas de destilação de hidrocarbonetos pesados, ou quando há forte não-idealidade líquida, modelos de atividade (como NRTL/UNIFAC) são requeridos.<br>
        O range de temperatura de colunas de destilação atmosférica é de aproximadamente 100°C (fração leve) a 370°C (fração pesada).
    </p>
</div>
""")

def plotar_comparativo(fluido_nome, T_c):

    p_eixo, z_rk, z_srk, z_pr = calcular_modelos_puros(fluido_nome, T_c)
    
    
    fig = go.Figure()
    fig.add_vrect(x0=0.1, x1=5, fillcolor="gray", opacity=0.06, layer="below", line_width=0,
                  annotation_text="🔵 Superfície", annotation_position="top left", annotation_font_size=11)
    
    fig.add_vrect(x0=20, x1=40, fillcolor="green", opacity=0.06, layer="below", line_width=0,
                  annotation_text="🟢 Reservatório Convencional", annotation_position="top left", annotation_font_size=11)
    
    fig.add_vrect(x0=70, x1=100, fillcolor="purple", opacity=0.06, layer="below", line_width=0,
                  annotation_text="🟣 Pré-Sal (HPHT)", annotation_position="top left", annotation_font_size=11)

    fig.add_trace(go.Scatter(x=p_eixo, y=z_rk, name='RK (Redlich-Kwong)', line=dict(color='#d62728', width=3)))
    fig.add_trace(go.Scatter(x=p_eixo, y=z_srk, name='SRK (Soave-RK)', line=dict(color='#2ca02c', width=3)))
    fig.add_trace(go.Scatter(x=p_eixo, y=z_pr, name='PR (Peng-Robinson)', line=dict(color='#1a2a40', width=3)))
    
    fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.3)
    
    
    fig.update_layout(
    
        height=700,
        width=1100,
        title=dict(text=f"Análise PVT: {fluido_nome} a {T_c}°C", x=0.5, font=dict(size=24, family='Segoe UI, serif')),
        xaxis_title="Pressão (MPa)",
        xaxis=dict(gridcolor="#e1e8ed", zeroline=False),
      
        yaxis_title="Fator de Compressibilidade (Z)",
        yaxis=dict(gridcolor="#e1e8ed", title_font_size=16, tickfont_size=14),
        template="simple_white",
        margin=dict(t=120, b=80, l=120, r=80),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(size=13),
                    title_font_size=14, bordercolor="#dce4ec", borderwidth=1, bgcolor="#fffcf5")
    )
    
    z_final = z_pr[-1]
    if z_final < 1:
        titulo, cor_hex = "🟡 DOMÍNIO DE ATRAÇÃO", "#d4a017"
        exp = "As forças intermoleculares de Van der Waals predominam, reduzindo o volume real. Típico de pressões moderadas."
    else:
        titulo, cor_hex = "🔴 DOMÍNIO DE REPULSÃO", "#b22222"
        exp = "O efeito de co-volume e repulsão molecular domina a alta pressão (HPHT), impedindo compressão."

    with caixa_diagnostico:
        clear_output()
        display(widgets.HTML(f"""
            <div style="border: 2px solid {cor_hex}; padding: 15px; border-radius: 8px; background-color: #fffdfc; color: {cor_hex};">
                <b style="font-size: 1.1em; color: {cor_hex};">{titulo}:</b> {exp}
            </div>
        """))

    fig.show()

In [77]:
# 7 
# Gráfico primeiro, depois o diagnóstico, depois a teoria fixa
interact(plotar_comparativo, 
         fluido_nome=widgets.Dropdown(options=list(fluidos.keys()), description='Fluido:'),
         T_c=widgets.IntSlider(min=-100, max=500, step=5, value=25, description='Temp (°C):'))

display(caixa_diagnostico)
display(teoria_z)

interactive(children=(Dropdown(description='Fluido:', options=('metano', 'etano', 'propano', 'n-butano', 'i-bu…

Output()

HTML(value='\n<div style="background-color: #f7f9fc; padding: 25px; border-radius: 12px; margin-top: 25px; fon…